# 05 — RQ2 Correlation Analysis

This notebook continues the analysis after:

- **Notebook 01**, which created the merged DiveFace and CR-FIQA dataset;
- **Notebook 02**, which explored the dataset and its distributions;
- **Notebook 03**, which established the predictive baselines;
- **Notebook 04**, which optimized the strongest machine-learning models.

## Research question

> **RQ2: Which facial and image characteristics are associated with the CR-FIQA score?**

The purpose of this notebook is **statistical association analysis**, not predictive
model optimization. It examines each characteristic separately and reports:

- Pearson correlations;
- Spearman rank correlations;
- point-biserial correlations for binary characteristics;
- mean and median differences for binary characteristics;
- Cohen's \(d\);
- Welch's t-tests;
- Mann–Whitney U tests;
- Benjamini–Hochberg false-discovery-rate correction;
- focused visualizations of the most relevant associations.

`age` is reported separately as a protected numeric characteristic. Demographic
group comparisons and group-specific consistency analyses are handled in later
notebooks.

## Main outputs

The notebook saves its results to:

```text
results/05_rq2_correlation_analysis/
├── figures/
└── tables/
```

## 1. Shared project setup

In [ ]:
from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(f"- {path}" for path in setup_candidates)
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep all notebooks in the same notebooks/ directory or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")
get_ipython().run_line_magic("run", f'"{SETUP_NOTEBOOK}"')

## 2. Imports and configuration

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=UserWarning)

ALPHA = 0.05
FDR_METHOD = "fdr_bh"
TOP_CONTINUOUS_PLOTS = 3

TARGET = "cr_fiqa_score"
IDENTITY_COLUMN = "cls"
IMAGE_PATH_COLUMN = "index"
DEMOGRAPHIC_GROUP_COLUMN = "group"

## 3. Input and output paths

In [ ]:
DATA_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

RESULTS_PATH = (
    PROJECT_PATH
    / "results"
    / "05_rq2_correlation_analysis"
)
FIGURES_PATH = RESULTS_PATH / "figures"
TABLES_PATH = RESULTS_PATH / "tables"

for path in [RESULTS_PATH, FIGURES_PATH, TABLES_PATH]:
    path.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Merged dataset not found. Run Notebook 01 first:\n"
        f"{DATA_FILE}"
    )

print(f"Dataset:          {DATA_FILE}")
print(f"Results folder:   {RESULTS_PATH}")

## 4. Load the merged dataset

Notebook 02 already contains the full exploratory analysis. Therefore, this
notebook performs only the checks needed for the correlation analysis.

In [ ]:
df = pd.read_csv(DATA_FILE)

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")

## 5. Feature definitions

In [ ]:
continuous_features = [
    "smile",
    "moustache",
    "beard",
    "sideburns",
    "head_roll",
    "head_yaw",
    "head_pitch",
    "blur",
    "exposure",
    "noise",
]

binary_features = [
    "mask",
    "headWear",
    "glasses",
    "eye_makeup",
    "lip_makeup",
    "forehead_occluded",
    "eye_occluded",
    "mouth_occluded",
]

protected_numeric_features = ["age"]

primary_features = continuous_features + binary_features
all_numeric_features = primary_features + protected_numeric_features

required_columns = list(
    dict.fromkeys(
        all_numeric_features
        + [
            TARGET,
            IDENTITY_COLUMN,
            IMAGE_PATH_COLUMN,
            DEMOGRAPHIC_GROUP_COLUMN,
        ]
    )
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

print(f"Primary characteristics:       {len(primary_features)}")
print(f"Protected numeric variables:   {protected_numeric_features}")

### Feature-type note

The annotation file contains a mixture of continuous, graded, and binary
variables. `moustache`, `beard`, and `sideburns` are retained in the
continuous/graded block because their observed values are validated rather than
assumed to be strictly binary. The explicitly binary variables are tested
against the expected values 0 and 1.

## 6. Data validation

In [ ]:
analysis_df = df.copy()

for column in all_numeric_features + [TARGET]:
    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="coerce",
    )

binary_validation_rows = []

for feature in binary_features:
    observed_values = sorted(
        analysis_df[feature].dropna().unique().tolist()
    )
    valid_binary = set(observed_values).issubset({0, 1})

    binary_validation_rows.append(
        {
            "Feature": feature,
            "Observed_Values": observed_values,
            "Valid_0_1_Coding": valid_binary,
            "Missing": int(analysis_df[feature].isna().sum()),
        }
    )

binary_validation = pd.DataFrame(binary_validation_rows)

invalid_binary_features = binary_validation.loc[
    ~binary_validation["Valid_0_1_Coding"],
    "Feature",
].tolist()

if invalid_binary_features:
    raise ValueError(
        "Unexpected values were found in binary feature(s): "
        f"{invalid_binary_features}"
    )

display(binary_validation)

In [ ]:
feature_support = pd.DataFrame(
    {
        "Feature": all_numeric_features,
        "Non_Missing": [
            int(analysis_df[feature].notna().sum())
            for feature in all_numeric_features
        ],
        "Missing": [
            int(analysis_df[feature].isna().sum())
            for feature in all_numeric_features
        ],
        "Unique_Values": [
            int(analysis_df[feature].nunique(dropna=True))
            for feature in all_numeric_features
        ],
    }
)

constant_features = feature_support.loc[
    feature_support["Unique_Values"] <= 1,
    "Feature",
].tolist()

valid_primary_features = [
    feature
    for feature in primary_features
    if feature not in constant_features
]

valid_all_features = [
    feature
    for feature in all_numeric_features
    if feature not in constant_features
]

if constant_features:
    print("Excluded constant features:", constant_features)
else:
    print("No constant features detected.")

display(feature_support)

Missing observations are handled **pairwise** in each statistical test. This
avoids removing a row merely because an unrelated characteristic is missing and
also makes the sample size for every result explicit.

## 7. Descriptive statistics and binary support

In [ ]:
descriptive_statistics = (
    analysis_df[valid_all_features + [TARGET]]
    .describe()
    .T
)

descriptive_statistics["missing"] = (
    analysis_df[valid_all_features + [TARGET]]
    .isna()
    .sum()
)

descriptive_statistics["unique_values"] = (
    analysis_df[valid_all_features + [TARGET]]
    .nunique()
)

display(descriptive_statistics)

In [ ]:
valid_binary_features = [
    feature
    for feature in binary_features
    if feature in valid_primary_features
]

binary_prevalence = pd.DataFrame(
    {
        "Feature": valid_binary_features,
        "N": [
            int(analysis_df[feature].notna().sum())
            for feature in valid_binary_features
        ],
        "Count_0": [
            int((analysis_df[feature] == 0).sum())
            for feature in valid_binary_features
        ],
        "Count_1": [
            int((analysis_df[feature] == 1).sum())
            for feature in valid_binary_features
        ],
        "Prevalence_1": [
            analysis_df[feature].mean()
            for feature in valid_binary_features
        ],
    }
)

display(binary_prevalence)

## 8. Correlation helper functions

In [ ]:
def correlation_strength(coefficient):
    """Return a descriptive magnitude label for a correlation."""
    if pd.isna(coefficient):
        return "not estimable"

    absolute_value = abs(coefficient)

    if absolute_value < 0.10:
        return "negligible"
    if absolute_value < 0.30:
        return "weak"
    if absolute_value < 0.50:
        return "moderate"
    if absolute_value < 0.70:
        return "strong"
    return "very strong"


def calculate_correlations(data, features, target, method):
    """Calculate pairwise correlations and FDR-adjusted p-values."""
    result_rows = []

    for feature in features:
        pair_data = data[[feature, target]].dropna()
        x = pair_data[feature]
        y = pair_data[target]

        if len(pair_data) < 3 or x.nunique() <= 1 or y.nunique() <= 1:
            coefficient = np.nan
            p_value = np.nan
        elif method == "pearson":
            coefficient, p_value = stats.pearsonr(x, y)
        elif method == "spearman":
            coefficient, p_value = stats.spearmanr(x, y)
        else:
            raise ValueError(
                "method must be either 'pearson' or 'spearman'."
            )

        if feature in binary_features:
            feature_type = "binary"
        elif feature in protected_numeric_features:
            feature_type = "protected_numeric"
        else:
            feature_type = "continuous_or_graded"

        result_rows.append(
            {
                "Feature": feature,
                "Method": method.capitalize(),
                "Feature_Type": feature_type,
                "N": len(pair_data),
                "Coefficient": coefficient,
                "Absolute_Coefficient": (
                    abs(coefficient)
                    if pd.notna(coefficient)
                    else np.nan
                ),
                "P_Value": p_value,
                "Direction": (
                    "positive"
                    if pd.notna(coefficient) and coefficient > 0
                    else "negative"
                    if pd.notna(coefficient) and coefficient < 0
                    else "none"
                ),
                "Strength": correlation_strength(coefficient),
            }
        )

    results = pd.DataFrame(result_rows)

    valid_p_values = results["P_Value"].notna()
    adjusted_p_values = np.full(len(results), np.nan)
    rejected = np.full(len(results), False, dtype=bool)

    if valid_p_values.any():
        correction = multipletests(
            results.loc[valid_p_values, "P_Value"],
            alpha=ALPHA,
            method=FDR_METHOD,
        )

        rejected[valid_p_values] = correction[0]
        adjusted_p_values[valid_p_values] = correction[1]

    results["Adjusted_P_Value_FDR"] = adjusted_p_values
    results["Significant_Raw"] = results["P_Value"] < ALPHA
    results["Significant_FDR"] = rejected

    return (
        results
        .sort_values(
            "Absolute_Coefficient",
            ascending=False,
            na_position="last",
        )
        .reset_index(drop=True)
    )

## 9. Pearson correlations

Pearson correlation measures linear association. For a variable coded as 0 and
1, Pearson correlation is mathematically equivalent to the point-biserial
correlation.

In [ ]:
pearson_results_all = calculate_correlations(
    analysis_df,
    valid_all_features,
    TARGET,
    method="pearson",
)

pearson_results_primary = (
    pearson_results_all.loc[
        pearson_results_all["Feature"].isin(
            valid_primary_features
        )
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    pearson_results_primary.style.format(
        {
            "Coefficient": "{:.4f}",
            "Absolute_Coefficient": "{:.4f}",
            "P_Value": "{:.4g}",
            "Adjusted_P_Value_FDR": "{:.4g}",
        }
    )
)

In [ ]:
point_biserial_results = (
    pearson_results_all.loc[
        pearson_results_all["Feature"].isin(
            valid_binary_features
        )
    ]
    .copy()
    .rename(
        columns={"Coefficient": "Point_Biserial_R"}
    )
    .reset_index(drop=True)
)

display(point_biserial_results)

## 10. Spearman rank correlations

Spearman correlation measures monotonic association and is less sensitive than
Pearson correlation to skewed distributions, extreme values, and non-linear but
monotonic relationships.

In [ ]:
spearman_results_all = calculate_correlations(
    analysis_df,
    valid_all_features,
    TARGET,
    method="spearman",
)

spearman_results_primary = (
    spearman_results_all.loc[
        spearman_results_all["Feature"].isin(
            valid_primary_features
        )
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    spearman_results_primary.style.format(
        {
            "Coefficient": "{:.4f}",
            "Absolute_Coefficient": "{:.4f}",
            "P_Value": "{:.4g}",
            "Adjusted_P_Value_FDR": "{:.4g}",
        }
    )
)

## 11. Pearson–Spearman comparison

In [ ]:
correlation_comparison = (
    pearson_results_primary[
        [
            "Feature",
            "N",
            "Feature_Type",
            "Coefficient",
            "P_Value",
            "Adjusted_P_Value_FDR",
            "Significant_Raw",
            "Significant_FDR",
        ]
    ]
    .rename(
        columns={
            "N": "Pearson_N",
            "Coefficient": "Pearson_R",
            "P_Value": "Pearson_P_Value",
            "Adjusted_P_Value_FDR": "Pearson_FDR_P",
            "Significant_Raw": "Pearson_Significant_Raw",
            "Significant_FDR": "Pearson_Significant_FDR",
        }
    )
    .merge(
        spearman_results_primary[
            [
                "Feature",
                "N",
                "Coefficient",
                "P_Value",
                "Adjusted_P_Value_FDR",
                "Significant_Raw",
                "Significant_FDR",
            ]
        ].rename(
            columns={
                "N": "Spearman_N",
                "Coefficient": "Spearman_Rho",
                "P_Value": "Spearman_P_Value",
                "Adjusted_P_Value_FDR": "Spearman_FDR_P",
                "Significant_Raw": "Spearman_Significant_Raw",
                "Significant_FDR": "Spearman_Significant_FDR",
            }
        ),
        on="Feature",
        how="outer",
        validate="one_to_one",
    )
)

correlation_comparison["Same_Direction"] = (
    np.sign(correlation_comparison["Pearson_R"])
    == np.sign(correlation_comparison["Spearman_Rho"])
)

correlation_comparison["Both_Significant_FDR"] = (
    correlation_comparison["Pearson_Significant_FDR"]
    & correlation_comparison["Spearman_Significant_FDR"]
)

correlation_comparison["Maximum_Absolute_Correlation"] = (
    correlation_comparison[
        ["Pearson_R", "Spearman_Rho"]
    ]
    .abs()
    .max(axis=1)
)

correlation_comparison = (
    correlation_comparison
    .sort_values(
        "Maximum_Absolute_Correlation",
        ascending=False,
        na_position="last",
    )
    .reset_index(drop=True)
)

display(
    correlation_comparison.style.format(
        {
            "Pearson_R": "{:.4f}",
            "Pearson_P_Value": "{:.4g}",
            "Pearson_FDR_P": "{:.4g}",
            "Spearman_Rho": "{:.4f}",
            "Spearman_P_Value": "{:.4g}",
            "Spearman_FDR_P": "{:.4g}",
            "Maximum_Absolute_Correlation": "{:.4f}",
        }
    )
)

## 12. Binary group comparisons and effect sizes

Correlation coefficients alone can hide the practical size of a binary-group
difference. Therefore, every supported binary characteristic is also evaluated
using:

- group sizes, means, and medians;
- mean difference \(1-0\);
- Cohen's \(d\);
- Welch's unequal-variance t-test;
- Mann–Whitney U test;
- separate FDR correction for both test families.

In [ ]:
def pooled_cohens_d(group_0, group_1):
    """Calculate Cohen's d using the pooled standard deviation."""
    n_0 = len(group_0)
    n_1 = len(group_1)

    if n_0 < 2 or n_1 < 2:
        return np.nan

    variance_0 = group_0.var(ddof=1)
    variance_1 = group_1.var(ddof=1)

    pooled_variance = (
        ((n_0 - 1) * variance_0)
        + ((n_1 - 1) * variance_1)
    ) / (n_0 + n_1 - 2)

    pooled_sd = np.sqrt(pooled_variance)

    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan

    return (
        group_1.mean() - group_0.mean()
    ) / pooled_sd

In [ ]:
binary_rows = []

for feature in valid_binary_features:
    group_0 = analysis_df.loc[
        analysis_df[feature] == 0,
        TARGET,
    ].dropna()

    group_1 = analysis_df.loc[
        analysis_df[feature] == 1,
        TARGET,
    ].dropna()

    if len(group_0) < 2 or len(group_1) < 2:
        binary_rows.append(
            {
                "Feature": feature,
                "N_0": len(group_0),
                "N_1": len(group_1),
                "Supported": False,
            }
        )
        continue

    welch_result = stats.ttest_ind(
        group_0,
        group_1,
        equal_var=False,
        nan_policy="omit",
    )

    mann_whitney_result = stats.mannwhitneyu(
        group_0,
        group_1,
        alternative="two-sided",
    )

    binary_rows.append(
        {
            "Feature": feature,
            "N_0": len(group_0),
            "N_1": len(group_1),
            "Supported": True,
            "Mean_0": group_0.mean(),
            "Mean_1": group_1.mean(),
            "Median_0": group_0.median(),
            "Median_1": group_1.median(),
            "Mean_Difference_1_minus_0": (
                group_1.mean() - group_0.mean()
            ),
            "Cohens_D": pooled_cohens_d(
                group_0,
                group_1,
            ),
            "Welch_T": welch_result.statistic,
            "Welch_P_Value": welch_result.pvalue,
            "Mann_Whitney_U": mann_whitney_result.statistic,
            "Mann_Whitney_P_Value": mann_whitney_result.pvalue,
        }
    )

binary_comparisons = pd.DataFrame(binary_rows)

supported_binary_mask = binary_comparisons["Supported"].fillna(False)

for p_column, adjusted_column, significant_column in [
    (
        "Welch_P_Value",
        "Welch_FDR_P",
        "Welch_Significant_FDR",
    ),
    (
        "Mann_Whitney_P_Value",
        "Mann_Whitney_FDR_P",
        "Mann_Whitney_Significant_FDR",
    ),
]:
    binary_comparisons[adjusted_column] = np.nan
    binary_comparisons[significant_column] = False

    valid_test_mask = (
        supported_binary_mask
        & binary_comparisons[p_column].notna()
    )

    if valid_test_mask.any():
        correction = multipletests(
            binary_comparisons.loc[
                valid_test_mask,
                p_column,
            ],
            alpha=ALPHA,
            method=FDR_METHOD,
        )

        binary_comparisons.loc[
            valid_test_mask,
            adjusted_column,
        ] = correction[1]

        binary_comparisons.loc[
            valid_test_mask,
            significant_column,
        ] = correction[0]

binary_comparisons["Absolute_Cohens_D"] = (
    binary_comparisons["Cohens_D"].abs()
)

binary_comparisons = (
    binary_comparisons
    .sort_values(
        "Absolute_Cohens_D",
        ascending=False,
        na_position="last",
    )
    .reset_index(drop=True)
)

display(binary_comparisons)

## 13. Protected numeric characteristic: age

In [ ]:
age_pearson = (
    pearson_results_all.loc[
        pearson_results_all["Feature"].isin(
            protected_numeric_features
        )
    ]
    .copy()
)

age_spearman = (
    spearman_results_all.loc[
        spearman_results_all["Feature"].isin(
            protected_numeric_features
        )
    ]
    .copy()
)

age_results = age_pearson.merge(
    age_spearman,
    on="Feature",
    suffixes=("_Pearson", "_Spearman"),
    validate="one_to_one",
)

display(age_results)

Age is kept outside the primary RQ2 feature ranking because it is a protected
characteristic. Its association is still documented transparently and saved as
a separate result.

## 14. Correlation overview figures

In [ ]:
pearson_plot_data = (
    pearson_results_primary
    .sort_values("Coefficient")
)

plt.figure(figsize=(10, 7))
plt.barh(
    pearson_plot_data["Feature"],
    pearson_plot_data["Coefficient"],
)
plt.axvline(0, linewidth=1)
plt.xlabel("Pearson correlation with CR-FIQA score")
plt.ylabel("Characteristic")
plt.title("Pearson Correlations — RQ2")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "pearson_correlations.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
spearman_plot_data = (
    spearman_results_primary
    .sort_values("Coefficient")
)

plt.figure(figsize=(10, 7))
plt.barh(
    spearman_plot_data["Feature"],
    spearman_plot_data["Coefficient"],
)
plt.axvline(0, linewidth=1)
plt.xlabel("Spearman correlation with CR-FIQA score")
plt.ylabel("Characteristic")
plt.title("Spearman Correlations — RQ2")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "spearman_correlations.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
heatmap_data = (
    correlation_comparison
    .set_index("Feature")[
        ["Pearson_R", "Spearman_Rho"]
    ]
)

fig, ax = plt.subplots(figsize=(7, 9))
image = ax.imshow(
    heatmap_data.values,
    aspect="auto",
    vmin=-1,
    vmax=1,
)

ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns)
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

for row_index in range(heatmap_data.shape[0]):
    for column_index in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[
            row_index,
            column_index,
        ]

        label = (
            f"{value:.2f}"
            if pd.notna(value)
            else "NA"
        )

        ax.text(
            column_index,
            row_index,
            label,
            ha="center",
            va="center",
        )

ax.set_title("Pearson and Spearman Correlations")
fig.colorbar(
    image,
    ax=ax,
    label="Correlation coefficient",
)
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "pearson_spearman_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 15. Focused continuous-feature visualizations

Only the strongest continuous or graded characteristics are plotted. This keeps
the notebook readable while preserving the complete numerical tables for all
characteristics.

In [ ]:
continuous_plot_candidates = (
    correlation_comparison.loc[
        correlation_comparison["Feature"].isin(
            continuous_features
        )
    ]
    .sort_values(
        "Maximum_Absolute_Correlation",
        ascending=False,
    )
)

top_continuous_features = (
    continuous_plot_candidates
    .head(
        min(
            TOP_CONTINUOUS_PLOTS,
            len(continuous_plot_candidates),
        )
    )["Feature"]
    .tolist()
)

print(
    "Selected continuous or graded characteristics:",
    top_continuous_features,
)

In [ ]:
for feature in top_continuous_features:
    pair_data = analysis_df[
        [feature, TARGET]
    ].dropna()

    result_row = correlation_comparison.loc[
        correlation_comparison["Feature"] == feature
    ].iloc[0]

    x = pair_data[feature]
    y = pair_data[TARGET]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        x,
        y,
        alpha=0.20,
        s=18,
    )

    if x.nunique() > 1:
        slope, intercept = np.polyfit(x, y, 1)
        x_values = np.linspace(
            x.min(),
            x.max(),
            100,
        )
        plt.plot(
            x_values,
            intercept + slope * x_values,
            linestyle="--",
            linewidth=2,
        )

    annotation = (
        f"N = {len(pair_data)}\n"
        f"Pearson r = {result_row['Pearson_R']:.3f}\n"
        f"Pearson FDR p = {result_row['Pearson_FDR_P']:.3g}\n"
        f"Spearman rho = {result_row['Spearman_Rho']:.3f}\n"
        f"Spearman FDR p = {result_row['Spearman_FDR_P']:.3g}"
    )

    plt.text(
        0.02,
        0.98,
        annotation,
        transform=plt.gca().transAxes,
        verticalalignment="top",
        bbox={
            "boxstyle": "round",
            "facecolor": "white",
            "alpha": 0.85,
        },
    )

    plt.xlabel(feature)
    plt.ylabel("CR-FIQA score")
    plt.title(f"{feature} vs. CR-FIQA Score")
    plt.tight_layout()
    plt.savefig(
        FIGURES_PATH
        / f"focused_continuous_{feature}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## 16. Focused binary-feature visualizations

A binary characteristic is selected for a boxplot when its point-biserial
correlation or Welch test remains significant after FDR correction. Features
without sufficient observations in both states remain documented in the support
and comparison tables but are not plotted.

In [ ]:
significant_binary_correlations = set(
    pearson_results_primary.loc[
        (
            pearson_results_primary["Feature"].isin(
                valid_binary_features
            )
            & pearson_results_primary["Significant_FDR"]
        ),
        "Feature",
    ]
)

significant_binary_group_tests = set(
    binary_comparisons.loc[
        binary_comparisons["Welch_Significant_FDR"],
        "Feature",
    ]
)

selected_binary_features = sorted(
    significant_binary_correlations
    | significant_binary_group_tests
)

print(
    "Binary characteristics selected for boxplots:",
    selected_binary_features,
)

In [ ]:
for feature in selected_binary_features:
    group_0 = analysis_df.loc[
        analysis_df[feature] == 0,
        TARGET,
    ].dropna()

    group_1 = analysis_df.loc[
        analysis_df[feature] == 1,
        TARGET,
    ].dropna()

    comparison_row = binary_comparisons.loc[
        binary_comparisons["Feature"] == feature
    ].iloc[0]

    correlation_row = pearson_results_primary.loc[
        pearson_results_primary["Feature"] == feature
    ].iloc[0]

    plt.figure(figsize=(7, 5))
    plt.boxplot(
        [group_0, group_1],
        tick_labels=["0", "1"],
        showfliers=False,
    )

    annotation = (
        f"N0 = {len(group_0)}, N1 = {len(group_1)}\n"
        f"Mean 0 = {comparison_row['Mean_0']:.3f}\n"
        f"Mean 1 = {comparison_row['Mean_1']:.3f}\n"
        f"Difference = "
        f"{comparison_row['Mean_Difference_1_minus_0']:.3f}\n"
        f"Cohen's d = {comparison_row['Cohens_D']:.3f}\n"
        f"Welch FDR p = {comparison_row['Welch_FDR_P']:.3g}\n"
        f"Point-biserial r = "
        f"{correlation_row['Coefficient']:.3f}"
    )

    plt.text(
        0.02,
        0.98,
        annotation,
        transform=plt.gca().transAxes,
        verticalalignment="top",
        bbox={
            "boxstyle": "round",
            "facecolor": "white",
            "alpha": 0.85,
        },
    )

    plt.xlabel(f"{feature} (0 = absent, 1 = present)")
    plt.ylabel("CR-FIQA score")
    plt.title(f"CR-FIQA Score by {feature}")
    plt.tight_layout()
    plt.savefig(
        FIGURES_PATH / f"focused_binary_{feature}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

### Binary effect-size overview

In [ ]:
binary_effect_plot_data = (
    binary_comparisons.loc[
        binary_comparisons["Supported"]
    ]
    .sort_values("Cohens_D")
    .copy()
)

plt.figure(figsize=(9, 6))
plt.barh(
    binary_effect_plot_data["Feature"],
    binary_effect_plot_data["Cohens_D"],
)
plt.axvline(0, linewidth=1)

for row_index, row in (
    binary_effect_plot_data
    .reset_index(drop=True)
    .iterrows()
):
    marker = (
        "*"
        if row["Welch_Significant_FDR"]
        else ""
    )

    horizontal_alignment = (
        "left"
        if row["Cohens_D"] >= 0
        else "right"
    )

    text_x = (
        row["Cohens_D"] + 0.002
        if row["Cohens_D"] >= 0
        else row["Cohens_D"] - 0.002
    )

    plt.text(
        text_x,
        row_index,
        marker,
        va="center",
        ha=horizontal_alignment,
    )

plt.xlabel("Cohen's d")
plt.ylabel("Binary characteristic")
plt.title("Binary Feature Effect Sizes on CR-FIQA Score")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "binary_feature_effect_sizes.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print(
    "* indicates significance after Welch-test FDR correction."
)

## 17. Focused summary tables

In [ ]:
focused_continuous_summary = (
    correlation_comparison.loc[
        correlation_comparison["Feature"].isin(
            top_continuous_features
        ),
        [
            "Feature",
            "Pearson_N",
            "Pearson_R",
            "Pearson_P_Value",
            "Pearson_FDR_P",
            "Spearman_Rho",
            "Spearman_P_Value",
            "Spearman_FDR_P",
            "Both_Significant_FDR",
            "Maximum_Absolute_Correlation",
        ],
    ]
    .sort_values(
        "Maximum_Absolute_Correlation",
        ascending=False,
    )
    .reset_index(drop=True)
)

focused_binary_summary = (
    binary_comparisons.loc[
        binary_comparisons["Feature"].isin(
            selected_binary_features
        ),
        [
            "Feature",
            "N_0",
            "N_1",
            "Mean_0",
            "Mean_1",
            "Mean_Difference_1_minus_0",
            "Cohens_D",
            "Welch_P_Value",
            "Welch_FDR_P",
            "Welch_Significant_FDR",
            "Mann_Whitney_P_Value",
            "Mann_Whitney_FDR_P",
            "Mann_Whitney_Significant_FDR",
        ],
    ]
    .assign(
        Absolute_Cohens_D=lambda table: (
            table["Cohens_D"].abs()
        )
    )
    .sort_values(
        "Absolute_Cohens_D",
        ascending=False,
    )
    .drop(columns="Absolute_Cohens_D")
    .reset_index(drop=True)
)

print("Focused continuous-feature summary:")
display(focused_continuous_summary)

print("Focused binary-feature summary:")
display(focused_binary_summary)

## 18. Automated RQ2 summary

In [ ]:
rq2_summary = correlation_comparison.copy()

rq2_summary["Robust_Association"] = (
    rq2_summary["Same_Direction"]
    & rq2_summary["Both_Significant_FDR"]
)

rq2_summary["Interpretation"] = np.select(
    [
        rq2_summary["Robust_Association"],
        rq2_summary[
            [
                "Pearson_Significant_FDR",
                "Spearman_Significant_FDR",
            ]
        ].any(axis=1),
    ],
    [
        (
            "Pearson and Spearman agree in direction and remain "
            "significant after FDR correction."
        ),
        (
            "Significant after FDR correction in one method only."
        ),
    ],
    default=(
        "No statistically significant association after "
        "FDR correction."
    ),
)

display(rq2_summary)

In [ ]:
significant_pearson = pearson_results_primary.loc[
    pearson_results_primary["Significant_FDR"],
    "Feature",
].tolist()

significant_spearman = spearman_results_primary.loc[
    spearman_results_primary["Significant_FDR"],
    "Feature",
].tolist()

robust_associations = rq2_summary.loc[
    rq2_summary["Robust_Association"],
    "Feature",
].tolist()

print("Significant Pearson associations after FDR:")
print(significant_pearson)

print("\nSignificant Spearman associations after FDR:")
print(significant_spearman)

print("\nRobust associations in both methods:")
print(robust_associations)

## 19. Interpretation guidance

When reporting RQ2, distinguish clearly between:

- **direction**: positive or negative association;
- **magnitude**: the absolute coefficient or effect size;
- **statistical evidence**: the FDR-adjusted p-value;
- **practical relevance**: whether the observed magnitude is meaningful;
- **robustness**: whether Pearson and Spearman agree;
- **support**: the number of usable observations or binary states.

A small coefficient can be statistically significant in a large dataset.
Therefore, significance must not be described as a large or practically
important effect. These analyses are univariate and do not establish causality
or control for other characteristics. Multivariable regression is performed in
Notebook 06.

## 20. Save all outputs

In [ ]:
binary_validation.to_csv(
    TABLES_PATH / "binary_feature_validation.csv",
    index=False,
)

feature_support.to_csv(
    TABLES_PATH / "feature_support.csv",
    index=False,
)

descriptive_statistics.to_csv(
    TABLES_PATH / "descriptive_statistics.csv",
)

binary_prevalence.to_csv(
    TABLES_PATH / "binary_feature_prevalence.csv",
    index=False,
)

pearson_results_all.to_csv(
    TABLES_PATH / "pearson_correlations_all_features.csv",
    index=False,
)

pearson_results_primary.to_csv(
    TABLES_PATH / "pearson_correlations_primary.csv",
    index=False,
)

spearman_results_all.to_csv(
    TABLES_PATH / "spearman_correlations_all_features.csv",
    index=False,
)

spearman_results_primary.to_csv(
    TABLES_PATH / "spearman_correlations_primary.csv",
    index=False,
)

point_biserial_results.to_csv(
    TABLES_PATH / "point_biserial_correlations.csv",
    index=False,
)

correlation_comparison.to_csv(
    TABLES_PATH / "pearson_spearman_comparison.csv",
    index=False,
)

binary_comparisons.to_csv(
    TABLES_PATH / "binary_feature_comparisons.csv",
    index=False,
)

age_results.to_csv(
    TABLES_PATH / "protected_age_correlation.csv",
    index=False,
)

focused_continuous_summary.to_csv(
    TABLES_PATH / "focused_continuous_summary.csv",
    index=False,
)

focused_binary_summary.to_csv(
    TABLES_PATH / "focused_binary_summary.csv",
    index=False,
)

rq2_summary.to_csv(
    TABLES_PATH / "rq2_summary.csv",
    index=False,
)

print("All Notebook 05 tables were saved.")

## 21. Saved-file summary

In [ ]:
print("Notebook 05 result files:")

for file_path in sorted(RESULTS_PATH.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(RESULTS_PATH))

## 22. Transition to Notebook 06

Notebook 05 identifies **univariate associations** between individual facial or
image characteristics and CR-FIQA scores.

Notebook 06 extends this analysis with multivariable regression in order to
estimate adjusted feature effects while accounting for the other included
characteristics. The exported correlation and binary-comparison tables provide
the unadjusted reference results for that comparison.